# TomatoRPE quickstart

Reproduce selected results from the reference tables. Select a molecular dynamics system to view its time series, then export the data and figure or verify the original trajectory.

[Source code and local instructions](https://github.com/mybagi/TomatoRPE) · [Zenodo trajectories](https://doi.org/10.5281/zenodo.24425383)


In [ ]:
# @title Set up TomatoRPE { display-mode: "form" }
# @markdown Load the repository and prepare the reference tables. Google Colab provides the plotting dependencies.
from pathlib import Path
import csv
import importlib.util
import json
import subprocess
import sys

from IPython.display import FileLink, display
import matplotlib.pyplot as plt

working_dir = Path.cwd()
candidates = (working_dir, working_dir.parent, working_dir / "TomatoRPE")
repository = next(
    (path for path in candidates if (path / "assets" / "systems.tsv").is_file()),
    None,
)
if repository is None:
    repository = working_dir / "TomatoRPE"
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/mybagi/TomatoRPE.git", str(repository)],
        check=True,
    )

assets = repository / "assets"
reference = Path(
    subprocess.check_output([sys.executable, "src/prepare_data.py"], cwd=repository, text=True).strip()
)
with (assets / "systems.tsv").open(encoding="utf-8", newline="") as handle:
    catalog = {row["system"]: row for row in csv.DictReader(handle, delimiter="\t")}

print(f"TomatoRPE is ready with {len(catalog)} indexed systems.")


In [ ]:
# @title Reproduce the summaries { display-mode: "form" }
# @markdown Check the reference files and reproduce selected transcriptomic, docking, and molecular dynamics summaries.
summary = json.loads(
    subprocess.check_output([sys.executable, "src/reproduce.py"], cwd=repository, text=True)
)
ranking_table = reference / "manuscript_tables" / "Supplementary_Data_3_435_root_priority_sensitivity.tsv"
with ranking_table.open(encoding="utf-8-sig", newline="") as handle:
    ranking = list(csv.DictReader(handle, delimiter="\t"))
rpe = next(row for row in ranking if row["Solyc_ID"] == "Solyc03g115820")

print(f"{summary['verified_files']} reference files passed checksum validation.")
print(f"The root interaction analysis identified {summary['transcriptomics']['significant_root_interactions']} loci.")
print(f"Ribulose-5-phosphate 3-epimerase ranked {rpe['equal_rank']} of {len(ranking)} with equal weights.")
print(f"The docking summary includes {summary['docking']['pairs']} scored compounds.")


In [ ]:
# @title Inspect a trajectory { display-mode: "form" }
# @markdown Select a system to plot its archived measurements over time. For the unliganded system, the plot reports root mean square displacement of core Cα atoms.
SYSTEM = "CNP0184442.0" # @param ["apo","CNP0184442.0","CNP0196064.0","CNP0211069.0","CNP0511659.1","CNP0542861.1","substrate","native_D-Ru5P_no-Zn_rep1","native_D-Ru5P_no-Zn_rep2","native_D-Ru5P_no-Zn_rep3","native_D-Ru5P_with-Zn_rep1","native_D-Ru5P_with-Zn_rep2","native_D-Ru5P_with-Zn_rep3"]
selected = catalog[SYSTEM]
observables = reference / "md_observables" / f"md_metrics_{SYSTEM}.tsv"
with observables.open(encoding="utf-8", newline="") as handle:
    rows = list(csv.DictReader(handle, delimiter="\t"))

distance_column = "fixed_pocket_heavy_minimum_image_distance_nm"
is_liganded = distance_column in rows[0]
measurement = distance_column if is_liganded else "core_ca_rmsd_nm"
times = [float(row["time_ns"]) for row in rows]
values = [10 * float(row[measurement]) for row in rows]

figure, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(times, values, color="#207C83", linewidth=1.25)
title = selected["ligand_or_state"]
if selected["replicate"]:
    condition = "with zinc" if selected["catalytic_metal"] == "Zinc" else "without zinc"
    title = f"{title} {condition} (replicate {selected['replicate']})"
ax.set(xlabel="Time (ns)", xlim=(0, 100), title=title)
if is_liganded:
    ax.set_ylabel("Minimum pocket distance (Å)")
    ax.axvspan(50, 100, color="#207C83", alpha=0.08)
    for cutoff in (4, 5):
        ax.axhline(cutoff, color="#586B76", linestyle="--", linewidth=0.8, label=f"{cutoff} Å")
    ax.legend(frameon=False)
    counts = summary["molecular_dynamics"][SYSTEM]
    print(f"{counts['frames_below_4_angstrom']} of {counts['frames_50_to_100_ns']} frames are within 4 Å from 50 to 100 ns.")
else:
    ax.set_ylabel("Protein core Cα RMSD (Å)")
ax.spines[["top", "right"]].set_visible(False)
figure.tight_layout()
plt.show()


In [ ]:
# @title Export the results { display-mode: "form" }
# @markdown Enable the export to save the selected time series as CSV, its figure as PNG, and the study summary as JSON.
EXPORT_RESULTS = False # @param {type:"boolean"}
OUTPUT_DIRECTORY = "" # @param {type:"string"}

if EXPORT_RESULTS:
    output = Path(OUTPUT_DIRECTORY).expanduser() if OUTPUT_DIRECTORY.strip() else repository.parent / "TomatoRPE_results"
    output.mkdir(parents=True, exist_ok=True)
    table_path = output / f"{selected['system']}.csv"
    with table_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    figure_path = output / f"{selected['system']}.png"
    figure.savefig(figure_path, dpi=180)
    summary_path = output / "tomatorpe_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    try:
        from google.colab import files
    except ImportError:
        for path in (table_path, figure_path, summary_path):
            display(FileLink(str(path)))
    else:
        for path in (table_path, figure_path, summary_path):
            files.download(str(path))


In [ ]:
# @title Verify the original trajectory { display-mode: "form" }
# @markdown Enable verification after selecting a system. Set a directory that contains the Zenodo files or enable the download for that system.
VERIFY_TRAJECTORY = False # @param {type:"boolean"}
DOWNLOAD_FROM_ZENODO = False # @param {type:"boolean"}
TRAJECTORY_DIRECTORY = "" # @param {type:"string"}

if VERIFY_TRAJECTORY:
    downloads = Path(TRAJECTORY_DIRECTORY).expanduser().resolve() if TRAJECTORY_DIRECTORY.strip() else repository.parent / "TomatoRPE_trajectories"
    system = selected["system"]
    if DOWNLOAD_FROM_ZENODO:
        subprocess.run(
            [sys.executable, "src/download_trajectory.py", "--system", system, "--output", str(downloads)],
            cwd=repository,
            check=True,
        )
    if importlib.util.find_spec("MDAnalysis") is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-r", "src/requirements/md.txt"],
            cwd=repository,
            check=True,
        )
    subprocess.run(
        [sys.executable, "src/verify_md_trajectory.py", "--trajectory-dir", str(downloads), "--system", system],
        cwd=repository,
        check=True,
    )
